# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdPQ1327iRfwVV7jXSriw7TtLu+U7bp9jKxl1/1ZKTu2f5+CgKslhTpEqQclSf//vNDAASIEFZybb37p3aF0siBhjMDOYb2qdWkMTz8L632rSOWCtcrpI0Y4mYxOptyou3eR7Oig9ZuCyfiEWehVH5bJFyfxbG95N4niZLlm1W8IGpp5erLExiP+qys1BkXXYSBvDvIN6o0Ss/W0ThVA+/go+TeBK/Yic840HGhvE6TJN4yePsiP2SJPcRZ8dJ5E/ZWrCzJPABkdORd3x5NvjA+rCXHs7Y419hNdGetPZhxxkAT1od5sczFidZfdDx0QRfrQ4uPeNzoINIojX3ZsljHCX+jKfeLEzbHbb3MxNZejSJGbwmLfofvT/ZxP4yBHSiDYsArYwLje5JGq45K6diMBVsLUk3XSbyFe4bCSYW8CbIM9GVM/b0F3uZn97zTOxNN3vhTOKWI1W7tCOx8FM+Y/MkgrkFe1yEsOZ97qc+bJzjzHP4RshJYSi7GH4eXrPHNMyAMAoZwDpLbITTJMl69Y2Gc6KhprkiBb6mvuBIJoMN/lTg37b+DE+BTLzteYiT53U6JXjKszyNC9C/JmHc1lN2AQWDFyLz71HgJMMQ+BV702NXabj00w0LFjx4OGokYNvaph7E1CCWxFLAFGrqaw/JATsrJWp/hvD7DasY5KqImzlhxyBflm6MT/iaJymcvBkLiSoRQKMQNsEX83i4msEEoqQJ1cVZO3VIxVoNFwpajqZzLYQvpEUY59yBRm07+gW0S0M4Hn12y3tR8sjxYOFeeXWncuk79zSArR9v2g8IpKfEWeiLW5AXP/MnLZSc1H9Ub/g8jDnw5q7DYCSC8x6PZ+IxBCGdtHrhahNPUVUU6KiZm/aPLyW4Elv3MJFPawyh4bZgk0A37LUiRTCjVGg2t/Drl1GFUfUx/GvAV6Bw6Q9ol505/gLgyheiPKaHPXaMx5MFfpzEqDDx3SwEbvFCMYZaXRWPSFrKeWun8HxDp3nfpGX3m8azPTZSJ3k76Ij0Lb0X1noSRkkrClCAAlRuwaBLnZ9BlW3u8wOmNjIkCT+Sgg3cx5mGm1KiVsPvXeJD3zcw3sBZq3U5DeoMPHLVp01EBynv1qaqDoZBdy8LMiJgD9pBiqVANmxN0qHGDWPNoBTmtz024n4aLJh7pyQFoGWUcQbXwZe2HvUNe10Kz2s543JD4C4zU9Cuyawo2JctCmq0TUXTOoALkDyKvFVVeymArpzNLXz0SCv4Hu1dK1pL5TWwudmASGa4D4JEt0GNKsT+kafCmPb/zukoidRwSvRrR52Pr/qpcWxd0kvx4OXjKwd+pyV5Vxw+qZGlP6dwFIaSdh0nU4k3nylzlu87WE0zFHDO02VC/UOOmEOotzOpgUH6BQ5EHWs5eAePxunNAC5uf+ZFJGnMdwrRCZ/7eZSxOQRuU99yS0pvTkWDsy162eEIGFB1zSAfdQrBKwb/M1SFc1BFhXz7HvUulYqR4Pj1yeWXi7PLwcnw2js5vYYJm6LpSQw7rwSSZ5fHgzPvy+X1r6OrwfHQRifjyxUuzSPB1fgPg9FQLfOtQWd9KUuc9cwo0EUSAT9ILDoyTTHK0jwAEgDlrLDyxHRlrwdfKjjSCjahikgFZr4efjy9GJ7sCKSjGgPw/PLz6XDkgjemRuBlskYUTdjx5x3gsrUneFoFvRx/Kri+DTrJFko3nAzGg922KeM5ADkfjgcE9ufR5UUNTs1He+OZj1C9v4okLjg2XC34kqdwwil9xD6f28ySYtHAsorMKJ55oZxegdJed4VG5ulcxvh6MPrkjcaDX04vftlxgiz1xcLKiMhBTYR1TFFQVj7bSl97cjeVx8PzK7V0ZTXg9/XpZ00bm8cq7Qf2IA2neYYH6uqGHUe5yMB1/j2EjXn8IODA0U5tnm0hWV2QTFodf7q5+HW0jeTGI9IEhIVBLTXDbpQ2oI/PbkZj92ExqRvI/VsgOOXw2rmmMStCPybpA0/tBc8H9Ofj6VlN5VWglz4uXfJVbbgZ8bpwmOjb4Fs2UVunuhV7wJYNOWeqbqsqdlrkjilXnqc+ehFKVLyTm+vB+PTywhsNj2G1w3cHvYPCtwCnlC3DOMd8xQpmeOCbeQqmhwl+jxls1k7AJVmC0pn6kR8HnCVzJh2QLIlAJ+FXv2fgzykr8vFmNGSn+5daUr2zIVql8en58PJmrHD4V43CK3gLS4GxmgkWcV9wNivQl/T2Pg0H1+MPw8G4Msm79zDJK3bupw9Mkhpwm0dgWGRakC3A186m3M/I1333XijCwcnHLC1l7sWRbQR1ip5mxEMbJKuN1gF/uRnegCb4sE36/5bznPdmU4PrTWB12bOAEddTldntMj969DeC5YB3RAiORids9JezMOOULcdkOI9C4CWmpaw9EUtwzB5A0qbmfhiBAyD+jZFZewxhVpzZ3qbL2TH2Ym/OcnGMUdUpyZ7FAlZnMQ+4EJj+NrJojPzrSYwsk76lqlZUlKKydvDGtNN1Z0LlxWxPoRxYmP8uKxlhKdC6voXvzOPpUA9dp7vWZZahLj4WxrfLauZUDyqRq+rw4hsTp0bFNYm1C21HhCCYS/+BAyNEe9aVTPCSh/44zblyPrdEKTpCeYUaKMjTlMfBBmLdpb9aJCBnoCG+yBN6BhKawdjzwX94mqUaP5CWt/LJ2eUXb3Q1RK7Wxxyg2jgA0ZiFsmh1c3F2en46Hp5gpleunqFG2hMrDspR+/BqVUXucr43VEJTg8BFVEjDk6JO2Cu20nYhDgSC5TxaznPOdJGAVgK08e8Ry2M8qKi4A4NcKPEC0QY9C8c5DYOHiJvYw+A12BMg/U5Y2htVCmUcLnmSZxS3jog8X/wsWMySe3adU82t2BxI4XVV56LiNoaUfBp/uh6OPl2enXgfroioBwfsB/j38F2P+IWff/2wL9zA1hpvDmkRi9ASl+vh+BqONRoxnPM0DrMQNOGP8BGCOlWheQuaMMNsWYZRTya0It1b8mUCumbsiwc2Tn2lCYHomGzaw3oxi1BdLiGaRl2E5eEMBlMi/06991CFWkQHc/GAgSE+9oIkj9EQg5TKdX+JkqmyJvA1BmVpEglD3uQTzw8ymUbFAzeJZYC008NGhGRdWPhIpb9ziiQpqMR/jrAm7KwNj9R4wTSALAykiRB7q8jP4NMSNrIEOxZOwyjMNiRMdoXSn/Ns03MUYWnxWgV10lqHM57osbQqBuE9kU/b6etbLHXv//Cno0nr33/+77vXXcK1S+M6bpDJRPxI41gxsIf+0qrdsbIA+OT2CETuDjGEDRN5OuxnFEOGJo2GaGpON6jqPBnBzHjEM96mEB2tugcHk9IPJXWnSRI5Su8faJpqvX2Ms7IPodas5wmmAsFASwNPzQcWQBiD0cfnkQ4LeREponSIlR8gcCoyZZFgcIx+3hJUTIwl9BQOBa4SZnKSaNNj40UIzmDK1zBEWLj9LYejhVIs8iXZgOZqe50qWL909DLUB3bqAvLRB068ZLccxtNlx/CVxyH4Wp6SmrmOSZ+whaSH/7xrd3oL/vX26Ke7Z+9Jo4xuI50hB9LPmgD4IlqCgIis6vQ5kDSQMVB8RfxHl9ZmeoJMlwynLokNeP7LEky2uvSQqw4suwZm1lo3YJLiByljcvJZKB62lf+MiSr5ShgoxartXEyxVOoyl2PBfPACTe8CdHHWBibdkhTKw3MHhivFNMQR+Bx5JNtk5PFkxExSW091EjwfsSf+bGV568nx+na3i2l9647xjfW5khQucjizwZUZ1PmQSiqAQCrOV3JVap/wQB/5GzIB4N4VXx8VvU63MOBO+Skdh8IaBX4s6s02rO3Ou8puBFqUQTiDJ5+CV6uvJwhA54PZxP4j3dqD0QgHNpY9LmXvEAuFyEFVwWQU0dwvEjheEJ+B3VfqjZqLwPYYig1mXEJsg7beobzd+qukEKwpfTfdiOVoIpIsqJZDjTkA4R2Kow7V6Khx6sVcurDsUnHUSV/04e2ZX2F0DJ5MLAxWznTkjNTQSTSTp1LMVhrSaoAA04xeAwxup5PWf8lQ94fJBOSj98O/oJWGEae/XFxeD48Ho2Gn2whapO8QGsVqG/ydUTlBbMGuCZ6uKcXApnyOPrQ6MEd64Om8CAbB/waKIm8WPvBVB+jIUyuLiKuLIOdkjqeo90ubjULg4hfqRW9Wa7NpTAhoOLewqNmqWomOf32RohresIZap1pulpPZZS9s0gKHsXz4Mztwq8bCUpdbbDLQ+qXMGWZh5sR/uUhXE88BUpqLY8nXO3ZNvJmxtR+Fs5KJxB1Dm4FhfVLzPhudegWvkOPflr3fgWUI5GSaazWDbc2LbeEdLdbMPfn4n84/XKZbkvR7eGjtvpmROKy09bs7GsWqysmQDZplwEhHnRRf6U3IeUiXUFHyQH5h1dmVijbWlCVoR/tjY79KreeRCFvMU5WlevcbYgQjER9bYdedIHgCXM+CRVti4epNcrEKa3TYIExmg1iCbURzCGSRXjjVc01k8eWMr4qd1cdrav9IaZzabBASPxQGVo+1xduB/rEamK/YkwJ61hZwul/6MY6ITHYg6+ztdR7LzSsDg426y2SWw3DVPI519DwmCQtVqteVdX3Bo4PlwPy1FjyC0E7lBRob5iFgcDfJkx5x9tJX2ufzNIrCKTi6oEgxVWt/zdM0SUvQfLpKE8zz7th9X3TbdwvndFKiJS8DaEcXdJd3v8q9MJ4nstMdodGXpRnuLEf2NBYrHkBAKyMmsOfp7BGby/E4XHw+PTkdUPkORwk7dQHTw+l8KoVm0sJlKTEARJ60jgFOJjgrjZjr1F96yymOOrAeECdTCa3XeXYGuDmGkCUZe6SJPPh6lWdtW+ZvJ614Hc5Cf08sQ1n539sDJqWbPcC3j/h2ZRasRzmyrsTCU1lFDYEZHj/rB2LdjZMFx+4HeANRaiYmrbuuvWbGv2Z9O4CpZFrolGGbD/qDK/1MaiHqPcqznlhFIewGiNcym6dVSoagwS712duqigLW3JrcwCCGht8e3DmHFhzBkXj05eg3d1UDKodrPpXzHt7tmA7Hdyo8w8m00IbCi9c8Djx/7Ydw3kF3u1NF1HQskAYXn4cXx2zB18H+4vAP78AiBAndnEDxVfcxQF8lGD9BwKIUgCnC3yJTsOv5fLni90oe9GrI+5LdXdCI8G3aN6Y5GX6+uDk7q8f6kxYiL/c9aSmmy7gIN1R5sGOoUga9QYRR/ydSf/IYOkLYQZ4lcbJMckHH/JqSqKmuS8SwRUMVCwhNma+u7Ui9ikaXA5MeVMx4EoogwZPDzn3Aex36+l7LibTbgFW4FIwym0yW0buKdYKGE1uNyFVuS1ByFS3fTBdhhXXlxVzFEb+ijHmgEsPM89qCR/OuEggvnG0P+PGFAD2tU+G5rWLNNBPIHD5BQTJh7NMIRmIV+QGHoy1VrJQpOOXGkzEXkV8+0AqCxOMCqGUm1mitYj+wePkexs910d17KhDUvYbPDdm92uwoCkZmUIoVaz/Z23ytd/n67rlTm0MW7D0wiW6SV4YrwTsiJVDk+a0hqq7lYZnBmNM2eA3TyzQclkMql53I0QQWg/+XtqVtRS5UW2CINZWWtk6XTnXhcVjunBkrODCohw2l6M7UsfJKEtIUpCMtUhpSK+X/GkyVYBIOYnA/UI6a9NPUsTHdNUtFkppUF6qsxAW+6jRytHIgnSzKqquLzu6SeltHp/vSmo1NJE0rb+lucaxfLm+YTrrRhYpZ0ebFRKn7hldjZzzdK0hWYOUzai5DzEhJwnnoT1p5Nt/7idqBwd3e0p5Nmag++bA9rKu15w0N9SBPmJ3D0BG209ZEQDnbdpcJwSi7CAyB95XO5UWWrZqvBeBLmSwA7aVSwcmG1jrEjg3uzjSwPPvlYfJj9NsC7mWJjB2UQQDks1xQmYraVcNZxG385cn4koZYYA8S8OzRQ9ybp5zrYqZ5wkqztPV8KS492VsqlDbeQTyqqPhudWyhnouxxTe1saaXvsVK1eAMt70KVvqPNShJVQKid7UB4KZkXtGVhAMxAOvhP+3q6cc+y/sQRZPPvNrgcuzzb9JeZmfIt2qvsr3tWzXXDqu+oLmMtZu1Ftq6RsXl1kim4cqclqoK8UgxeO2qJMGa7ogl1M/upB3hhTZSTanm/hE23MtkH3lt+VJ9akCiz3eoUdKds3y5auMxBey72GMDFr9/6MKzUaNTDUz6dgZOci/N+u5yNMTEQcOc1TTiLhPjy2GeNOQ2bV2W8crh9uidL+0ZTjlaCU9lTZQi5vFsBfKSkSrukn580bmzPaHKmJoKH+FlYTAE7NN4fMWuLkdjdJR/GY6ZwoSpHgKKYoIkSWfYQghjBlenNQWuco9VH7eSvTaNkf5S2l04EU8V4OcnTQTLFa/LF+DrKQNiT40vmaYQZFxaN4Kne4N7uv5whGtSHGfGh/vVAzlpPdsTYhkQFwtl9Ycat+o8NnAqjo+g89PpyVCvXR6+OrjC+hYxJOd6b7xZqSzGpOWvVhG2uAGj96W+qBEExtnZuN61ki/4WspTX+PY1cv11d8KQqRKKrPBR1Iu8LlL9gfC8/6b96RHUi5WDpLIq0GY50EbyPp9dnjgqifga5rMNvK2zQrvQ84gLJvxF6km904yVvh7oo1zGYNfOKBOGW3worQh9pBCKi4pYxIjb4MvHYpIEJ2MKQN4onKRMaBD6TposKa38je4tf83DpPTaUmpYE1AtoIE/9hfhfvK7O9rJshCb0Gceq4J+YHX1VRKRSg3XztnHZRI07kyajfULwfaqPTRlJpeYndhIHZTzU6BkGqY+gtXaXIPeEkEyx51UMMoFjVJKKVAYYH6++m5NuDWkoo7TdTyu28mueGpdlmd3jSBI8IgQvfV+Ar1u2Ww8b2cSx4qHKM8G+VFHImCl8zjFYQ2KoOHPmPMv2asSMzKrB3oOhHe0+/51JijTZc0bRb1SrT+VPCgXzc7O7BFL2LXAOuEWviC1qtHouWVxxeVnU4+lgTtsr8mU0oeAg5dSRP4+FV9Bovgye/QzVKODHU/YV0ZTj5YYJAEJNFB72DLCcmn2BquaK7QwCItmN8w4Ducj9+kJeUecaB8VxtQ7BvHFB9qw2xy4Fj7mxqASSpyV8zPv0VjWpx0H+FvOnh+gKa0pjClmyN3qORF5ijto9fM+PLz8CsP6CaSrs3t4Zp4ywiT4bIKkqV+LHBFLHeUN2EwMsCrSYRG11RSMYZA2PMaQjAUlApXdGU3rGwgY3inUSxgmd7yYd2QZnfhjHcpZVlWCm6fdn8rL1magmCYP5QIJ0xVeO5MVs29YH6vxioO6auYnqqLApOfng3+SlEu59dCbsxbSHI5ypD0O7MHxt3EUiFAVfWU9XXT/75jMtpjoyRPA61oi14rmhoZfCLbqZ8qi9SbB6olIXt7Kptfa+Ot4l7CyYQ1PK/90IErnQ3bC2PvqVzMRlAOl1W3nSZDOajNZmc35pjaaBdodstFqhxw9LJ+W4rWaGr9rfnK8hn9/Nkxnloksr5tWLY2g5eNB9s4ydVOIpt33ZJlHXudwx77kIfRjH38iFVNQ4OAhlyCyjPWgBnopoS8w6HOnFaHxkPpycR5ZCeNg3ReA4PvvLUfIcRbM5Gy9hCHoDae6oXymVzlU+/wD+/ZPvs0/HxsN5UJj8rBqJzhr6yf6mmpejZACOPbEhZ9Baq6YhXcUZGunHu1VF2yinkcbF97fnov05GtveBoLbdjFnypvkwdmeqHClbv1JfB3+R4zLcDBau/qGbcNXx5RQhovxJ95Mzp3Jy4hsKapxuUPktLOtb7vs0bZfD/zc2//57NG3K9nMl5rcaAOMEWY902slF/qfGkOIt2M4d1vn7XL45QtWVtOQNVlYHpxmaEvbXE2YA1mzWswdY0P0iK1DyuPT+22nT0t+sknHkxv/fxtpWXCV0ke+De33mauGB0LKcYGK740ZvauEIzG6nqio5622NnPsRRC62kVEuFTBdQ2FhGiuqCmuECN9lYlQCWP0TAXj0Vlv0ZfaU/J1P4TvoEz2htt5S4e72epXsQP7uH5IoSRcAN6gzBPJH58PRquKVjxOormeZzdH/7bywaydrJ1KN7en13LQRr5N4qUKFG5fv5Sji/l5dE+9T5VP5IDL7cv5kkL5jHRIGe3KnjMNKoPv2pd0PpF7Y3yQFGLRHw7H/vT4wZG6WYqy2nV81VOO3tmzuNT0Nm7Tf/wBaPnPsiOm/ZmcWMbXjvvCJ6USgiXr6NonHy2CRP5gtWaOPQPUsSqSntDQjVFirRoaqklZ6ah+PrxXC0DrBTeFoHg9NCAOrkvDgehKsYD+9fHE8MLSDo08swRR110tI1LFftS7+eG1iGr4raAA5WwxM4xY8+yNmW3HG1T7wAfAijqCowjcq4mG2W4+XiojxXv4+Gr6aABpuycWn5nNzY3+GdePflyjIowAHVnn/jKUT4VRlu2kjNjcZfk4BT24Y1sUN3hgFbBUdskdpxg/QzeRQYlH1wKg+XJfr3qiu//GyE1kYBt/prWpWwu9Nc3MVOT0/f5KpG61SOxRFW11clOilI2zUm23rl0CiblhDd6up1KXUXTqv47DalErCKEBkb2Hrl0dqoEa2qZJYR/GoBrGICUU4bfzNA/XSA5W+4hXGSH/7x4L3LwSkSOiIP0NfAawQb1n7S6Bz13syf2fmHTo/J1CP+zKl2c0xxfN9jF0kWzjeUf6zkv+10qdLVhv6tErtbkKOyyh968t6Bin+pye47A363j7BjGsBmaTXg/8ayN76sO65l2jDNZfqj3nREyXhVNuOUEQwp1Zes6rlfkgryEEBk/njoEphJPvvp7Qz+5T8dqIZe7P9VbZ3YX4o/0IUN1EfKAS7KY88Vl1fNCO9V1/DpiYYxU/oNMLiq/E8ElEs5fe16r6mqoeGz8w/s8/XgvNO4jEyTjmRXOKzz+j+5YO1P+rbDoMyodl6r0L7ej04/fPD6ImFtvNrwUf3UZOe1a3cW+SvHo7E8VJSBTBD5HwKwOmJt+aKkkW68ZnQTaw7h16LevHBzfVZTay93LhR4lyN0qt3Vl+p26V9eoi6fAfw7f3NQ7A3UFm3puNySFhmjc6KxOi5J6CpbO71nZzrBZDBo2bfzt+wL+En6DlyN4qodEP7v6ARkbaPXtFMNJmuobxGbR4mDBzioPTVORT69iDhftd83DCkvx1UF7bAHKoscDSrkOSuDFbWLPyyjpMWoSdbNK5VHGgTPbPNWGfmGkVtoVDrNrhuZ0hxqTK3yDS7nAnFgVm/B2QWz4tBXAQ1eHfSq3GqQUHdgZYUPcj1XgGAs+K74sRzQOp7qaV+CfLe1cVLWqG/dMdGctc2tunyD5s2+dfor30wT0MKn+ANBab4yswbFOQNECnM1e6ONlLI2oYaEwzXd4A+2peC7gKNHhxJ/Uah6sBQyWxiifjFPKeIQr46gFfI8WfPziA6eV6ToahSaxK3n/wFpJBbH"
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.

import time
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main()
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
